# Predial Vision MX — Final: Teacher + Temporal Training
ResNet101 trained on 5x data (original + 4 Wayback dates 2018-2024)
Same buildings, different conditions → robust model
Target: F1 > 0.70

In [ ]:
# Cell 1: Imports + STRICT GPU guard
import gc
import json
import math
import os
import sys
import urllib.request
from pathlib import Path

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from shapely.geometry import shape, box
from shapely.ops import unary_union
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation import deeplabv3_resnet101

# --- STRICT GPU guard ---
if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Aborting.')

device_cap = torch.cuda.get_device_capability()
sm = device_cap[0] * 10 + device_cap[1]
print(f'GPU: {torch.cuda.get_device_name(0)}  SM: {sm}')
if sm < 75:
    raise SystemExit(f'GPU SM={sm} < 75. P100 or older not supported. Need T4/A100/V100 (SM>=75).')

DEVICE = torch.device('cuda')
print(f'Using device: {DEVICE}  ({torch.cuda.get_device_name(0)})')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Cell 2: Configuration
WORKING_DIR = Path('/kaggle/working')
INPUT_DIR   = Path('/kaggle/input')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Data sources
MS_BUILDINGS_URL = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.2.0-data/ms_buildings_nr.geojson.gz'
WAYBACK_URLS = {
    '2024': 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.0-wayback/wayback_2024.npz',
    '2022': 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.0-wayback/wayback_2022.npz',
    '2020': 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.0-wayback/wayback_2020.npz',
    '2018': 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.0-wayback/wayback_2018.npz',
}

# Training config
CHIP_SIZE     = 512
STRIDE_TRAIN  = 256
BATCH_SIZE    = 8
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 40
TOTAL_EPOCHS  = EPOCHS_STAGE1 + EPOCHS_STAGE2  # 50
LR_STAGE1     = 1e-3
LR_STAGE2     = 5e-5
POS_WEIGHT    = 5.0
NUM_WORKERS   = 2

print('Configuration:')
print(f'  CHIP_SIZE={CHIP_SIZE}, STRIDE_TRAIN={STRIDE_TRAIN}, BATCH_SIZE={BATCH_SIZE}')
print(f'  Stage1 epochs={EPOCHS_STAGE1} (head only, lr={LR_STAGE1})')
print(f'  Stage2 epochs={EPOCHS_STAGE2} (full finetune, lr={LR_STAGE2})')
print(f'  Total epochs={TOTAL_EPOCHS}')

In [ ]:
# Cell 3: Download MS Buildings from GitHub Release
ms_gz_path  = WORKING_DIR / 'ms_buildings_nr.geojson.gz'
ms_json_path = WORKING_DIR / 'ms_buildings_nr.geojson'

if not ms_json_path.exists():
    print('Downloading MS Buildings...')
    urllib.request.urlretrieve(MS_BUILDINGS_URL, ms_gz_path)
    import gzip, shutil
    with gzip.open(str(ms_gz_path), 'rb') as f_in:
        with open(str(ms_json_path), 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f'MS Buildings saved to {ms_json_path}')
else:
    print(f'MS Buildings already exists: {ms_json_path}')

In [ ]:
# Cell 4: Load original imagery from Kaggle dataset
import glob

# Find nicolas-romero.tif in Kaggle input
tif_candidates = list(INPUT_DIR.rglob('nicolas-romero.tif'))
if not tif_candidates:
    tif_candidates = list(INPUT_DIR.rglob('*.tif'))
if not tif_candidates:
    raise FileNotFoundError(f'No .tif found under {INPUT_DIR}')

TIF_PATH = tif_candidates[0]
print(f'Found imagery: {TIF_PATH}')

with rasterio.open(TIF_PATH) as src:
    IMG_CRS       = src.crs
    IMG_TRANSFORM = src.transform
    IMG_BOUNDS    = src.bounds
    IMG_H         = src.height
    IMG_W         = src.width
    n_bands       = src.count
    print(f'Image: {IMG_W}x{IMG_H}, bands={n_bands}, CRS={IMG_CRS}')
    print(f'Bounds: {IMG_BOUNDS}')

    # Read first 3 bands as HxWx3 uint8
    bands = src.read([1, 2, 3])  # 3xHxW
    IMG_ARRAY = np.moveaxis(bands, 0, -1).astype(np.uint8)  # HxWx3

print(f'IMG_ARRAY shape: {IMG_ARRAY.shape}, dtype: {IMG_ARRAY.dtype}')
print(f'Value range: [{IMG_ARRAY.min()}, {IMG_ARRAY.max()}]')

In [ ]:
# Cell 5: Download 4 Wayback mosaics from GitHub Release
for year, url in WAYBACK_URLS.items():
    npz_path = WORKING_DIR / f'wayback_{year}.npz'
    if not npz_path.exists():
        print(f'Downloading wayback_{year}.npz ...')
        urllib.request.urlretrieve(url, npz_path)
        size_mb = npz_path.stat().st_size / 1e6
        print(f'  Saved {size_mb:.1f} MB')
    else:
        print(f'  wayback_{year}.npz already exists')

# Quick shape verification for each wayback file
print('\nVerifying wayback shapes:')
for year in WAYBACK_URLS:
    npz_path = WORKING_DIR / f'wayback_{year}.npz'
    data = np.load(str(npz_path), mmap_mode='r')
    shape_wb = data['imagery'].shape
    print(f'  {year}: {shape_wb}  (expected ~{IMG_H}x{IMG_W}x3)')
    del data

print('All wayback files verified.')

In [ ]:
# Cell 6: Rasterize MS Buildings to mask (chunked with geopandas)
print('Loading MS Buildings GeoJSON...')
gdf = gpd.read_file(str(ms_json_path))
print(f'Loaded {len(gdf)} building footprints, CRS={gdf.crs}')

# Reproject if needed
if gdf.crs != IMG_CRS:
    print(f'Reprojecting from {gdf.crs} to {IMG_CRS}...')
    gdf = gdf.to_crs(IMG_CRS)

# Clip to image bounds
img_box = box(IMG_BOUNDS.left, IMG_BOUNDS.bottom, IMG_BOUNDS.right, IMG_BOUNDS.top)
gdf = gdf[gdf.geometry.intersects(img_box)].copy()
print(f'Buildings within image bounds: {len(gdf)}')

# Rasterize
shapes = [(geom, 1) for geom in gdf.geometry if geom is not None and geom.is_valid]
MASK = rasterize(
    shapes,
    out_shape=(IMG_H, IMG_W),
    transform=IMG_TRANSFORM,
    fill=0,
    dtype=np.uint8
)
print(f'MASK shape: {MASK.shape}, building pixels: {MASK.sum():,} ({MASK.mean()*100:.2f}%)')

# Free GeoDataFrame
del gdf, shapes
gc.collect()

In [ ]:
# Cell 7: Create MULTI-TEMPORAL training dataset
# KEY INNOVATION: Same MS Buildings mask applies to all 5 dates.
# Model learns buildings under different lighting, shadows, seasons, sensors.

def extract_train_chips(img_hwc, mask, chip_size, stride, split_row):
    """Extract chips from training region (above split_row)."""
    chips_i, chips_m = [], []
    H, W = img_hwc.shape[:2]
    for y in range(0, min(H, split_row) - chip_size + 1, stride):
        for x in range(0, W - chip_size + 1, stride):
            c = img_hwc[y:y+chip_size, x:x+chip_size]
            m = mask[y:y+chip_size, x:x+chip_size]
            if c.mean() < 5:
                continue
            chips_i.append(c.copy())
            chips_m.append(m.copy())
    return chips_i, chips_m

split_row = int(IMG_H * 0.7)
all_train_chips = []
all_train_masks = []

# --- Original imagery ---
ci, cm = extract_train_chips(IMG_ARRAY, MASK, CHIP_SIZE, STRIDE_TRAIN, split_row)
all_train_chips.extend(ci)
all_train_masks.extend(cm)
print(f'Original: {len(ci)} train chips')

# --- Val chips (original only, to keep evaluation consistent) ---
val_chips, val_masks = [], []
for y in range(split_row, IMG_H - CHIP_SIZE + 1, CHIP_SIZE):
    for x in range(0, IMG_W - CHIP_SIZE + 1, CHIP_SIZE):
        c = IMG_ARRAY[y:y+CHIP_SIZE, x:x+CHIP_SIZE]
        m = MASK[y:y+CHIP_SIZE, x:x+CHIP_SIZE]
        if c.mean() < 5:
            continue
        val_chips.append(c.copy())
        val_masks.append(m.copy())
print(f'Validation: {len(val_chips)} chips (original only)')

# --- Wayback dates: load one at a time, extract chips, free immediately ---
for year in ['2024', '2022', '2020', '2018']:
    npz_path = WORKING_DIR / f'wayback_{year}.npz'
    data = np.load(str(npz_path))
    wb = data['imagery'].astype(np.uint8)  # HxWx3
    h = min(wb.shape[0], IMG_H)
    w = min(wb.shape[1], IMG_W)
    wb = wb[:h, :w]
    mask_crop = MASK[:h, :w]
    sr = min(split_row, h)

    ci, cm = extract_train_chips(wb, mask_crop, CHIP_SIZE, STRIDE_TRAIN, sr)
    all_train_chips.extend(ci)
    all_train_masks.extend(cm)
    print(f'{year}: {len(ci)} train chips')

    del wb, data, ci, cm, mask_crop
    gc.collect()

print(f'\nTotal training chips: {len(all_train_chips)} (5 dates)')
print(f'Total validation chips: {len(val_chips)} (original only)')

In [ ]:
# Cell 8: Dataset class + augmentations
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class BuildingDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images    = images
        self.masks     = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = self.images[idx]  # HxWx3 uint8
        mask = self.masks[idx]   # HxW uint8
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']   # 3xHxW float tensor
            mask = aug['mask'].float().unsqueeze(0)  # 1xHxW
        return img, mask


train_ds = BuildingDataset(all_train_chips, all_train_masks, transform=train_aug)
val_ds   = BuildingDataset(val_chips,       val_masks,       transform=val_aug)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# Cell 9: Build ResNet101 model
class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=5.0, smooth=1.0):
        super().__init__()
        self.bce   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss  = self.bce(logits, targets)
        probs     = torch.sigmoid(logits)
        inter     = (probs * targets).sum()
        dice_loss = 1 - (2 * inter + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return bce_loss + dice_loss


# Build model
model = deeplabv3_resnet101(weights='DEFAULT')
model.classifier[4]     = nn.Conv2d(256, 1, kernel_size=1)  # 256 channels for ResNet101
model.aux_classifier[4] = nn.Conv2d(256, 1, kernel_size=1)  # 256 channels for ResNet101
model = model.to(DEVICE)

criterion = DiceBCELoss(pos_weight=POS_WEIGHT).to(DEVICE)
scaler    = torch.cuda.amp.GradScaler()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: DeepLabV3 ResNet101  |  Total params: {total_params/1e6:.1f}M')

In [ ]:
# Cell 10: Two-stage training
def compute_f1(logits, targets, threshold=0.5):
    preds = (torch.sigmoid(logits) > threshold).float()
    tp = (preds * targets).sum().item()
    fp = (preds * (1 - targets)).sum().item()
    fn = ((1 - preds) * targets).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return f1, precision, recall


def run_epoch(loader, train_mode, optimizer=None):
    model.train(train_mode)
    total_loss, total_f1 = 0.0, 0.0
    for imgs, masks in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast():
            out    = model(imgs)
            logits = out['out']
            loss   = criterion(logits, masks)
            if 'aux' in out:
                aux_logits = out['aux']
                loss = loss + 0.4 * criterion(aux_logits, masks)
        if train_mode and optimizer is not None:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        with torch.no_grad():
            f1, _, _ = compute_f1(logits, masks)
        total_loss += loss.item()
        total_f1   += f1
    return total_loss / len(loader), total_f1 / len(loader)


history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_val_f1  = 0.0
best_model_path = WORKING_DIR / 'final_teacher_temporal_model.pth'

# ---- STAGE 1: Head only, backbone frozen ----
print('=== Stage 1: Head only (epochs 1-{}) ==='.format(EPOCHS_STAGE1))
for p in model.backbone.parameters():
    p.requires_grad = False

head_params = (list(model.classifier.parameters()) +
               list(model.aux_classifier.parameters()))
optimizer_s1 = optim.Adam(head_params, lr=LR_STAGE1)

for ep in range(1, EPOCHS_STAGE1 + 1):
    tr_loss, _        = run_epoch(train_loader, train_mode=True,  optimizer=optimizer_s1)
    vl_loss, vl_f1    = run_epoch(val_loader,   train_mode=False)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_f1'].append(vl_f1)
    if ep % 5 == 0 or ep == 1:
        print(f'Ep {ep:03d}/{TOTAL_EPOCHS} | tr_loss={tr_loss:.4f} vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f}')
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), best_model_path)

# ---- STAGE 2: Full finetune ----
print('\n=== Stage 2: Full finetune (epochs {}-{}) ==='.format(EPOCHS_STAGE1 + 1, TOTAL_EPOCHS))
for p in model.backbone.parameters():
    p.requires_grad = True

optimizer_s2 = optim.Adam(model.parameters(), lr=LR_STAGE2)
scheduler    = optim.lr_scheduler.CosineAnnealingLR(optimizer_s2, T_max=EPOCHS_STAGE2)

for ep in range(EPOCHS_STAGE1 + 1, TOTAL_EPOCHS + 1):
    tr_loss, _        = run_epoch(train_loader, train_mode=True,  optimizer=optimizer_s2)
    vl_loss, vl_f1    = run_epoch(val_loader,   train_mode=False)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_f1'].append(vl_f1)
    if ep % 5 == 0:
        print(f'Ep {ep:03d}/{TOTAL_EPOCHS} | tr_loss={tr_loss:.4f} vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f}')
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), best_model_path)

print(f'\nTraining complete. Best val F1: {best_val_f1:.4f}')
print(f'Best model saved to: {best_model_path}')

In [ ]:
# Cell 11: Training curves (save as PNG)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='coral')
axes[0].axvline(EPOCHS_STAGE1, color='gray', linestyle='--', label='Stage 1/2 split')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_f1'], label='Val F1', color='green')
axes[1].axhline(0.70, color='red', linestyle='--', label='Target F1=0.70')
axes[1].axvline(EPOCHS_STAGE1, color='gray', linestyle='--', label='Stage 1/2 split')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = WORKING_DIR / 'training_curves.png'
plt.savefig(str(curves_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved: {curves_path}')

In [ ]:
# Cell 12: TTA functions
def tta_predict_chip(model, chip_tensor):
    """
    Test-Time Augmentation: 8 augmentations (original + 3 rotations + flips).
    chip_tensor: 1x3xHxW float tensor on CPU
    Returns: HxW numpy probability map
    """
    preds = []
    # 4 rotations
    for k in range(4):
        aug = torch.rot90(chip_tensor, k, dims=[2, 3])
        aug = aug.to(DEVICE)
        with torch.cuda.amp.autocast():
            with torch.no_grad():
                out  = model(aug)['out']
                prob = torch.sigmoid(out).cpu()
        # Rotate back
        prob = torch.rot90(prob, -k, dims=[2, 3])
        preds.append(prob.squeeze().numpy())

    # Horizontal flip + 4 rotations
    flipped = torch.flip(chip_tensor, dims=[3])
    for k in range(4):
        aug = torch.rot90(flipped, k, dims=[2, 3])
        aug = aug.to(DEVICE)
        with torch.cuda.amp.autocast():
            with torch.no_grad():
                out  = model(aug)['out']
                prob = torch.sigmoid(out).cpu()
        prob = torch.rot90(prob, -k, dims=[2, 3])
        prob = torch.flip(prob, dims=[3])
        preds.append(prob.squeeze().numpy())

    return np.mean(preds, axis=0)  # HxW


def tta_sliding_window(model, img_hwc, chip_size=512, stride=256):
    """
    TTA sliding-window inference over a full image.
    img_hwc: HxWx3 uint8
    Returns: HxW probability map in [0,1]
    """
    norm_fn = A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    H, W   = img_hwc.shape[:2]
    prob_map   = np.zeros((H, W), dtype=np.float32)
    count_map  = np.zeros((H, W), dtype=np.float32)

    model.eval()
    for y in range(0, H - chip_size + 1, stride):
        for x in range(0, W - chip_size + 1, stride):
            chip = img_hwc[y:y+chip_size, x:x+chip_size]
            t    = norm_fn(image=chip)['image'].unsqueeze(0)  # 1x3xHxW
            prob = tta_predict_chip(model, t)  # HxW
            prob_map[y:y+chip_size, x:x+chip_size] += prob
            count_map[y:y+chip_size, x:x+chip_size] += 1.0

    count_map = np.where(count_map == 0, 1, count_map)
    return prob_map / count_map


print('TTA functions defined.')

In [ ]:
# Cell 13: TTA inference on original imagery
print('Loading best model for inference...')
model.load_state_dict(torch.load(str(best_model_path), map_location=DEVICE))
model.eval()

print(f'Running TTA sliding window inference on {IMG_W}x{IMG_H} image...')
PROB_MAP = tta_sliding_window(model, IMG_ARRAY, chip_size=CHIP_SIZE, stride=CHIP_SIZE // 2)

print(f'Probability map: min={PROB_MAP.min():.4f}, max={PROB_MAP.max():.4f}, mean={PROB_MAP.mean():.4f}')
print('Inference complete.')

In [ ]:
# Cell 14: Threshold sweep [0.20 - 0.50]
thresholds = np.arange(0.20, 0.51, 0.05)

print('Threshold sweep:')
print(f'{"Thr":>6}  {"F1":>8}  {"Prec":>8}  {"Rec":>8}')
print('-' * 38)

sweep_results = {}
prob_tensor = torch.from_numpy(PROB_MAP).unsqueeze(0).unsqueeze(0)  # 1x1xHxW
mask_tensor = torch.from_numpy(MASK.astype(np.float32)).unsqueeze(0).unsqueeze(0)

for thr in thresholds:
    preds = (prob_tensor > thr).float()
    tp    = (preds * mask_tensor).sum().item()
    fp    = (preds * (1 - mask_tensor)).sum().item()
    fn    = ((1 - preds) * mask_tensor).sum().item()
    prec  = tp / (tp + fp + 1e-8)
    rec   = tp / (tp + fn + 1e-8)
    f1    = 2 * prec * rec / (prec + rec + 1e-8)
    sweep_results[round(thr, 2)] = {'f1': f1, 'precision': prec, 'recall': rec}
    print(f'{thr:>6.2f}  {f1:>8.4f}  {prec:>8.4f}  {rec:>8.4f}')

BEST_THRESHOLD = max(sweep_results, key=lambda t: sweep_results[t]['f1'])
print(f'\nBest threshold: {BEST_THRESHOLD}  F1={sweep_results[BEST_THRESHOLD]["f1"]:.4f}')

In [ ]:
# Cell 15: Geometric filter
from rasterio.features import shapes as rasterio_shapes
from shapely.geometry import shape as shapely_shape
import pyproj

def rectangularity(geom):
    """Area / area_of_minimum_bounding_rectangle."""
    mbr = geom.minimum_rotated_rectangle
    if mbr is None or mbr.area == 0:
        return 0.0
    return geom.area / mbr.area

def compactness(geom):
    """4*pi*area / perimeter^2 (Polsby-Popper)."""
    p = geom.length
    if p == 0:
        return 0.0
    return 4 * math.pi * geom.area / (p ** 2)

# Binarise at best threshold
bin_map = (PROB_MAP > BEST_THRESHOLD).astype(np.uint8)

# Morphological cleanup
kernel  = np.ones((3, 3), np.uint8)
bin_map = cv2.morphologyEx(bin_map, cv2.MORPH_CLOSE, kernel)
bin_map = cv2.morphologyEx(bin_map, cv2.MORPH_OPEN,  kernel)

# Vectorise
raw_polys = []
for geom_dict, val in rasterio_shapes(bin_map, mask=bin_map, transform=IMG_TRANSFORM):
    if val == 1:
        raw_polys.append(shapely_shape(geom_dict))

print(f'Raw polygons before filter: {len(raw_polys)}')

# Determine UTM CRS for area computation
cx = (IMG_BOUNDS.left + IMG_BOUNDS.right)  / 2
cy = (IMG_BOUNDS.bottom + IMG_BOUNDS.top) / 2
utm_crs = pyproj.CRS.from_dict({'proj': 'utm', 'zone': int((cx + 180) / 6) + 1,
                                  'datum': 'WGS84', 'south': cy < 0})
proj_to_utm   = pyproj.Transformer.from_crs(IMG_CRS.to_epsg() or 4326, utm_crs, always_xy=True)

MIN_AREA_M2   = 20.0
MIN_RECT      = 0.45
MIN_COMPACT   = 0.10

filtered_polys = []
for poly in raw_polys:
    if not poly.is_valid:
        poly = poly.buffer(0)
    # Project to UTM for area check
    coords = np.array(poly.exterior.coords)
    xs, ys = proj_to_utm.transform(coords[:, 0], coords[:, 1])
    from shapely.geometry import Polygon
    poly_utm = Polygon(zip(xs, ys))
    if poly_utm.area < MIN_AREA_M2:
        continue
    if rectangularity(poly_utm) < MIN_RECT:
        continue
    if compactness(poly_utm) < MIN_COMPACT:
        continue
    filtered_polys.append(poly)

print(f'Filtered polygons: {len(filtered_polys)}')
n_polygons = len(filtered_polys)

In [ ]:
# Cell 16: Final evaluation + comparison table
# Rasterise filtered polygons back to compute pixel-level metrics
if filtered_polys:
    final_shapes = [(g, 1) for g in filtered_polys]
    final_mask_pred = rasterize(
        final_shapes,
        out_shape=(IMG_H, IMG_W),
        transform=IMG_TRANSFORM,
        fill=0,
        dtype=np.uint8
    )
else:
    final_mask_pred = np.zeros((IMG_H, IMG_W), dtype=np.uint8)

tp_f = float((final_mask_pred & MASK).sum())
fp_f = float((final_mask_pred & (1 - MASK)).sum())
fn_f = float(((1 - final_mask_pred) & MASK).sum())
final_p  = tp_f / (tp_f + fp_f + 1e-8)
final_r  = tp_f / (tp_f + fn_f + 1e-8)
best_f1  = 2 * final_p * final_r / (final_p + final_r + 1e-8)

print('COMPARISON ACROSS ALL PHASES:')
print('  v10 (OSM):          F1=0.176,  P=34.7%, R=11.8%,  Polygons=10')
print('  v12 (DiceBCE):      F1=0.49,   P=38%,   R=70-91%, Polygons=4,844')
print('  P1 (TTA):           F1=0.6041, P=46.3%, R=87%,    Polygons=4,519')
print('  P3 (Teacher):       F1=0.6818, P=54.2%, R=92%,    Polygons=8,340')
print(f'  FINAL (Teacher+5x): F1={best_f1:.4f}, P={final_p*100:.1f}%, R={final_r*100:.1f}%, Polygons={n_polygons}')

if best_f1 >= 0.70:
    print('\nTARGET ACHIEVED: F1 >= 0.70')
else:
    print('\nTarget F1=0.70 not yet reached. Consider more epochs or data augmentation.')

In [ ]:
# Cell 17: Save outputs
import json as json_mod

# --- 1. Model already saved as best during training ---
print(f'Model: {best_model_path}')

# --- 2. Save final_buildings.geojson ---
geojson_path = WORKING_DIR / 'final_buildings.geojson'
if filtered_polys:
    gdf_out = gpd.GeoDataFrame(geometry=filtered_polys, crs=IMG_CRS)
    gdf_out.to_file(str(geojson_path), driver='GeoJSON')
    print(f'GeoJSON saved: {geojson_path}  ({len(filtered_polys)} buildings)')
else:
    print('No filtered polygons to save.')

# --- 3. Save final_eval.json ---
eval_path = WORKING_DIR / 'final_eval.json'
eval_dict = {
    'model': 'DeepLabV3-ResNet101',
    'training_data': '5 dates (original + 2018/2020/2022/2024 Wayback)',
    'total_train_chips': len(all_train_chips),
    'total_val_chips': len(val_chips),
    'best_threshold': float(BEST_THRESHOLD),
    'final_f1': float(best_f1),
    'final_precision': float(final_p),
    'final_recall': float(final_r),
    'n_polygons': int(n_polygons),
    'epochs': TOTAL_EPOCHS,
    'best_val_f1_during_training': float(best_val_f1),
}
with open(str(eval_path), 'w') as f:
    json_mod.dump(eval_dict, f, indent=2)
print(f'Eval JSON saved: {eval_path}')

# --- 4. Visualization PNG (3 panels) ---
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Panel 1: Original imagery
axes[0].imshow(IMG_ARRAY)
axes[0].set_title('Original Imagery')
axes[0].axis('off')

# Panel 2: Ground truth mask
axes[1].imshow(MASK, cmap='gray')
axes[1].set_title('MS Buildings Ground Truth')
axes[1].axis('off')

# Panel 3: Predictions overlay
axes[2].imshow(IMG_ARRAY)
pred_overlay = np.zeros((*final_mask_pred.shape, 4), dtype=np.float32)
pred_overlay[final_mask_pred == 1] = [1.0, 0.2, 0.2, 0.5]  # red semi-transparent
axes[2].imshow(pred_overlay)
axes[2].set_title(f'Predictions (F1={best_f1:.4f}, N={n_polygons})')
axes[2].axis('off')

plt.suptitle('Predial Vision MX — Final Teacher + Temporal Training', fontsize=14)
plt.tight_layout()
vis_path = WORKING_DIR / 'final_visualization.png'
plt.savefig(str(vis_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Visualization saved: {vis_path}')

print('\nAll outputs saved:')
print(f'  {best_model_path}')
print(f'  {geojson_path}')
print(f'  {eval_path}')
print(f'  {vis_path}')